Funció per a poder llegir .json complexes

In [2]:
import json
from typing import List, Tuple
import polars as pl

In [3]:
def leer_json(file_path):
    data = []
    with open(file_path, 'r', encoding= 'utf-8') as file:
        
        for line in file:
            try:
                json_obj = json.loads(line)
                data.append(json_obj)
            
            except json.JSONDecodeError as e:
                print(f"Error al decodificar JSON: {e}")
                
    return data

## Generació de books.csv i users.csv
La informació s'extreu a partir dels arxius 'short_interactions.csv' i 'goodreads_book_genres_initial.json'

In [4]:
#!pip install polars

In [5]:
interactions = pl.read_csv('interactions.csv')
# books = pl.DataFrame(leer_json('books.json'))
generos_libros = pl.DataFrame(leer_json('genres.json'))

---

In [6]:
books_read_count = interactions.group_by("user_id").agg(pl.sum("is_read").alias("books_read"))
filtered_users = books_read_count.filter(books_read_count["books_read"] >= 10)["user_id"].unique().to_list()

interactions = interactions.filter(interactions['user_id'].is_in(filtered_users))

# Lista de 'book_id' del DataFrame interactions
interactions = interactions.with_columns(interactions['book_id'].cast(pl.Utf8))
book_ids = interactions['book_id'].unique().to_list()

# Filtrar generos_libros para mantener solo aquellos libros que estén en book_ids_filtrados
generos_libros = generos_libros.with_columns(generos_libros['book_id'].cast(pl.Utf8))
generos_libros = generos_libros.filter(generos_libros['book_id'].is_in(book_ids))

In [7]:
#convertir la columna 'generos' en una columna para cada genero
generes = []
for i in generos_libros['genres']:
    generes.append(list(i.values()))

generes = list(zip(*generes))

noms_generes = ['Fantasy', 'Fiction', 'Mystery', 'Poetry', 
           'History', 'Romance', 'Non-fiction', 
           'Children', 'Young-adult', 'Comics']


# Añadir las columnas al DataFrame
for lista, nombre in zip(generes, noms_generes):
    generos_libros = generos_libros.with_columns(pl.Series(nombre, lista))

generos_libros = generos_libros.drop('genres')

for columna in generos_libros.columns:
    generos_libros = generos_libros.with_columns(generos_libros[columna].fill_null(0))

In [8]:
generos_libros

book_id,Fantasy,Fiction,Mystery,Poetry,History,Romance,Non-fiction,Children,Young-adult,Comics
str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
"""1333909""",0,219,0,0,5,0,0,0,0,0
"""287140""",0,0,0,0,0,0,3,0,0,0
"""287141""",1,1,0,0,9,0,0,6,1,0
"""378460""",0,2,0,0,0,0,0,0,0,0
"""287149""",0,0,0,0,1,0,24,0,0,0
"""89371""",0,33,0,3,0,0,266,3,3,0
"""89373""",0,1760,31,0,12,0,0,0,0,0
"""89375""",0,16,0,0,178,0,534,0,0,6
"""89376""",0,0,0,0,0,0,163,0,0,0


In [9]:
llibres = []
for i in noms_generes:
    llibres.append(generos_libros[i].to_list())

llibres = list(zip(*llibres))

In [10]:
def normalize(libro:Tuple, total):
    norm = tuple(round(num/total,3) for num in libro)
    return norm

percentatges = []
for idx, llibre in enumerate(llibres):
    total = sum(llibre)
    if total == 0:
        percentatges.append((0 for _ in range(10)))
    else:
        percentatges.append(normalize(llibre, total))

percentatges

[(0.0, 0.978, 0.0, 0.0, 0.022, 0.0, 0.0, 0.0, 0.0, 0.0),
 (0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0),
 (0.056, 0.056, 0.0, 0.0, 0.5, 0.0, 0.0, 0.333, 0.056, 0.0),
 (0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0),
 (0.0, 0.0, 0.0, 0.0, 0.04, 0.0, 0.96, 0.0, 0.0, 0.0),
 (0.0, 0.107, 0.0, 0.01, 0.0, 0.0, 0.864, 0.01, 0.01, 0.0),
 (0.0, 0.976, 0.017, 0.0, 0.007, 0.0, 0.0, 0.0, 0.0, 0.0),
 (0.0, 0.022, 0.0, 0.0, 0.243, 0.0, 0.728, 0.0, 0.0, 0.008),
 (0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0),
 (0.0, 0.409, 0.0, 0.0, 0.318, 0.0, 0.0, 0.183, 0.09, 0.0),
 (0.0, 0.104, 0.0, 0.0, 0.0, 0.0, 0.016, 0.872, 0.008, 0.0),
 (0.0, 0.857, 0.0, 0.0, 0.0, 0.143, 0.0, 0.0, 0.0, 0.0),
 (0.0, 0.807, 0.0, 0.0, 0.0, 0.121, 0.0, 0.0, 0.072, 0.0),
 (0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0),
 (0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0),
 (0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0),
 (0.0, 0.284, 0.0, 0.062, 0.021, 0.0, 0.633, 0.0, 0.0, 0.0),
 (0.074, 0.894, 0.0, 0.0, 0.0, 

---

A partir de la base de dades de llibres, fem un nou dataset on s'agrupa per usuaris, i dona informació de:
- La quantitat de llibres que ha llegit
- La puntuació mitja del llibres que fa
- El percentatge de llibres que ha fet un review
- Per cada gènere:
    - Percentatge de llibres d'aquest caire
- 

In [11]:
percentatges = list(zip(*percentatges))

In [12]:
# Añadir las columnas al DataFrame
for lista, nombre in zip(percentatges, noms_generes):
    generos_libros = generos_libros.with_columns(pl.Series(nombre, lista))

In [13]:
generos_libros.write_csv('generesxbook.csv')

In [47]:
problem = interactions.group_by('user_id').agg([
    pl.col('book_id').alias('read_books'),
    pl.col('is_read').sum().alias('num_read'),
    pl.col('rating').mean().alias('mean_rating'),
    (pl.col('is_reviewed').sum() / pl.count()).alias('prop_reviwed')
])

In [48]:
problem

#es converteix la columna 'libros_leidos'  a una string amb tots els llibres separats per un espai
#això es fa perque sino no es pot guardar el CSV
# problem = problem.with_columns(
#     problem['read_books'].map_elements(lambda lst: " ".join(lst)).alias('read_books')
# )

user_id,read_books,num_read,mean_rating,prop_reviwed
i64,list[str],i64,f64,f64
787966,"[""536"", ""1313"", … ""7490""]",66,3.393939,0.075758
250500,"[""943"", ""400"", … ""375757""]",58,3.344828,0.051724
327317,"[""1313"", ""997"", … ""7319""]",74,3.648649,0.013514
152488,"[""909"", ""1405"", … ""270005""]",42,3.166667,0.071429
483592,"[""628823"", ""2183785"", … ""30059""]",37,4.054054,0.324324
494441,"[""645484"", ""17537"", … ""18070""]",30,4.466667,0.0
601004,"[""1003"", ""1203"", … ""38033""]",22,2.681818,0.090909
832384,"[""536"", ""197"", … ""454751""]",20,4.25,0.0
226128,"[""162596"", ""13568"", … ""312065""]",588,4.29932,0.083333


In [24]:
generos_libros.filter(generos_libros['book_id'].is_in(problem[0]['read_books'][0]))

book_id,Fantasy,Fiction,Mystery,Poetry,History,Romance,Non-fiction,Children,Young-adult,Comics
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""427""",0.0,0.044,0.0,0.0,0.211,0.0,0.744,0.0,0.0,0.0
"""557229""",0.0,0.165,0.138,0.0,0.0,0.0,0.0,0.656,0.042,0.0
"""1329176""",0.017,0.057,0.0,0.0,0.0,0.0,0.0,0.0,0.002,0.923
"""1102286""",0.05,0.225,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.725
"""1370""",0.0,0.035,0.0,0.0,0.918,0.0,0.047,0.0,0.0,0.0
"""1218934""",0.118,0.765,0.118,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""996""",0.0,0.0,0.0,0.0,0.091,0.0,0.909,0.0,0.0,0.0
"""16788""",0.661,0.3,0.008,0.03,0.0,0.0,0.0,0.0,0.0,0.0
"""24389""",0.026,0.211,0.0,0.0,0.0,0.0,0.0,0.711,0.053,0.0


In [53]:
c2 = []
for p in range(problem.shape[0]):
    libros = generos_libros.filter(generos_libros['book_id'].is_in(problem[p]['read_books'][0]))
    c = [libros[genere].mean() for genere in noms_generes]
    c2.append(c)

c2

KeyboardInterrupt: 

In [ ]:
perc_generes = list(zip(*c2))
for lista, nombre in zip(perc_generes, noms_generes):
    problem = problem.with_columns(pl.Series(nombre, lista))

problem

## Base de dades de llibres

Ens quedem només amb els llibres que han llegit els usuaris

In [33]:
libros = leer_json('books.json')
libros = pl.DataFrame(libros)
libros = libros.with_columns(libros['book_id'].cast(pl.Utf8))
libros = libros.filter(libros['book_id'].is_in(book_ids))
libros

isbn,text_reviews_count,is_ebook,average_rating,format,authors,num_pages,isbn13,book_id,ratings_count,title
str,str,str,str,str,list[struct[2]],str,str,str,str,str
"""0743509986""","""6""","""false""","""3.23""","""Audio CD""","[{""626222"",""""}]","""""","""9780743509985""","""1333909""","""10""","""Good Harbor"""
"""0850308712""","""5""","""false""","""3.40""","""""","[{""149918"",""""}]","""""","""9780850308716""","""287140""","""15""","""Runic Astrolog…"
"""1599150603""","""7""","""false""","""4.13""","""Paperback""","[{""3041852"",""""}]","""162""","""9781599150604""","""287141""","""46""","""The Aeneid for…"
"""0425040887""","""4""","""false""","""4.38""","""Paperback""","[{""215594"",""""}]","""""","""9780425040881""","""378460""","""12""","""The Wanting of…"
"""0922915113""","""39""","""false""","""3.81""","""Paperback""","[{""2983296"",""""}, {""40075"",""Foreword by""}]","""147""","""9780922915118""","""287149""","""986""","""The Devil's No…"
"""0413675106""","""2""","""false""","""3.81""","""""","[{""27397"",""""}]","""""","""9780413675101""","""89371""","""11""","""The Te Of Pigl…"
"""555118000X""","""19""","""false""","""3.82""","""""","[{""3083854"",""""}]","""""","""9785551180005""","""89373""","""77""","""The Bonfire of…"
"""0800759494""","""2885""","""false""","""3.91""","""""","[{""39054"",""""}, {""11099"",""""}]","""""","""9780800759490""","""89375""","""68157""","""90 Minutes in …"
"""0842379428""","""566""","""false""","""4.26""","""""","[{""4862"",""""}]","""533""","""9780842379427""","""89376""","""7345""","""Heaven"""


##### Variables eliminades:
- text_reviws_count -> quantitat de reviws que ha rebut el llibre per part dels ususaris
- series -> indica si a part del llibre s'ha generat una serie o peli a partir daquest
- popular_shelves -> a quins apartats de good read pertanyia
- asin -> codi asin del llibre, per reconeixe'l
- is_ebook -> si el llibre es troba en format ebook o no
- kindle_asin -> codi asin del llibre en format kindle, per reconeixe'l
- description -> descripció llarga sobre de que tracta el llibre
- link -> link al llibre 
- publisher -> la editorial del llibre
- publication_day -> dia en que es va publicar el llibre
- publication_month -> mes en que es va publicar el llibre
- edition_information -> informació adicional sobre la edició del llibre (pràcticament totes nul)
- url -> url per poder trobar el llibre
- image_url url per poder trobar la portada del llibre
- work_id -> identificador de obres (més general que els llibres)
- title_without_series -> títol a seques del llibre

Variables a estudiar:
- isbn -> conté els ibsn de 10 dígits dels llibres
-  country_code -> la nacionalitat d'on s'han extret les dades
- language_code -> en quin idioma està escrit el llibre original
- average_rating -> mitja de la puntuació al llibre
- similar_books -> aquells llibres de la base de dades que es considera que estan realcionats
- formal -> tipus de llibre (tapadura, blanda, de bolsillo...)
- authors -> autors que han creat el llibre (pot ser autor, ilustradors, autors de l'apendix....)
- num_pages -> pàgines del llibre
- publication_year -> any de la publicació del llibre
- isbn13 -> conté els ibsn de 13 dígits dels llibres
- book_id -> identificador del llibre
- ratings_count -> quantitat de persones que l'han puntuat
- title -> titol del llibre. Si pertany a una saga o serie també inclou el nom de la saga




In [156]:
#Columna a borrar
llibres = libros.drop('text_reviews_count')

Preprocessament

In [157]:
llibres

isbn,country_code,language_code,average_rating,similar_books,format,authors,num_pages,isbn13,publication_month,publication_year,book_id,ratings_count,work_id,title
str,str,str,str,list[str],str,list[struct[2]],str,str,str,str,str,str,str,str
"""0850308712""","""US""","""""","""3.40""",[],"""""","[{""149918"",""""}]","""""","""9780850308716""","""""","""""","""287140""","""15""","""278577""","""Runic Astrolog…"
"""037583687X""","""US""","""""","""3.98""","[""614054"", ""272343"", … ""1833502""]","""Hardcover""","[{""137561"",""""}]","""288""","""9780375836879""","""7""","""2006""","""89377""","""6949""","""86258""","""Penny from Hea…"
"""0590417010""","""US""","""eng""","""4.43""","[""834493"", ""452189"", … ""1081530""]","""Hardcover""","[{""5411"",""""}]","""40""","""9780590417013""","""9""","""1995""","""89378""","""1331""","""86259""","""Dog Heaven"""
"""0865165149""","""US""","""""","""4.50""",[],"""""","[{""166520"",""""}, {""166517"",""Editor""}, … {""3958429"",""Translator""}]","""""","""9780865165144""","""9""","""2002""","""287142""","""2""","""23097481""","""Servius' Comme…"
"""0553576348""","""US""","""""","""4.09""","[""644540"", ""2309183"", … ""2168992""]","""""","[{""51308"",""""}, {""51307"",""""}]","""""","""9780553576344""","""""","""""","""89379""","""627""","""86260""","""Hello from Hea…"
"""""","""US""","""eng""","""3.88""","[""3311228"", ""913719"", … ""722035""]","""Paperback""","[{""10885"",""""}]","""245""","""0345227018095""","""""","""""","""828189""","""64""","""1774048""","""Tarzan of the …"
"""0749729120""","""US""","""""","""3.66""",[],"""""","[{""10657"",""""}]","""93""","""9780749729127""","""""","""""","""388676""","""59""","""2055963""","""The Queen Eliz…"
"""0380821001""","""US""","""""","""3.81""","[""114555"", ""219231"", … ""622100""]","""Paperback""","[{""20560"",""""}]","""512""","""9780380821006""","""4""","""2003""","""89588""","""1480""","""822875""","""The Visitor"""
"""0953626954""","""US""","""en-GB""","""3.67""",[],"""Paperback""","[{""150591"",""""}]","""248""","""9780953626953""","""""","""2001""","""257654""","""27""","""249714""","""Gillian of the…"


In [158]:
#Authors ens quedem amb el primer, que és l'escriptor
autors = []
for author in llibres['authors']:
    if len(author) == 0:
        autors.append(None)
    else:
        autors.append(author[0]['author_id'])
      
llibres = llibres.with_columns(pl.Series('author_id', autors))
llibres = llibres.drop('authors')

In [159]:
#Diferències entre 'country_code' i 'language_code'
print('Num Country Code -> ',llibres['language_code'].unique().shape[0])
print('Num Laungage Code -> ' ,llibres['country_code'].unique().shape[0])
llibres = llibres.drop('country_code')

Num Country Code ->  54
Num Laungage Code ->  1


In [160]:
#Diferencies entre 'isbn' i 'isbn13'
print('No isbn13 -> ',llibres.filter(llibres['isbn13'] == "").shape[0])
print('No isbn ->', llibres.filter(llibres['isbn'] == "").shape[0])

#reemplaçar isbn13 d'aquells que no tenen per isbn
llibres = llibres.with_columns(
    pl.when(llibres['isbn13'] == "")
    .then(llibres['isbn'])
    .otherwise(llibres['isbn13'])
    .alias('isbn13')
)

print('No isbn amb el preprocessat ->', llibres.filter(llibres['isbn13'] == "").shape[0])
llibres = llibres.drop('isbn')

No isbn13 ->  644
No isbn -> 705


In [ ]:
#que els llibres recomanats siguin únicament dels que es troben a la base de dades
llibres_ids = llibres['book_id'].unique().to_list()

autors_sim = []
for similars in llibres['similar_books']:
    if len(similars) == 0:
        autors_sim.append(None)
    else:
        new = []
        for llibre in similars:
            if llibre in llibres_ids: new.append(llibre)
        autors_sim.append(new)

llibres = llibres.with_columns(pl.Series('similar_books', autors_sim))

#Converteix la llista de llibres comuns en una string amb tots els llibres comuns separats per un espai
#Es fa això perque sino no es pot guardar la base de dades
llibres = llibres.with_columns(
    llibres['similar_books'].apply(lambda lst: " ".join(lst)).alias('similar_books')
)

In [213]:
#Convertir els "" en Nones
for variable in llibres.columns:
    llibres = llibres.with_columns(
    pl.when(llibres[variable] == "")
    .then(None)
    .otherwise(llibres[variable])
    .alias(variable)
)

In [232]:
#Canviar average_rating i rating_count a numèriques
llibres = llibres.with_columns(llibres['average_rating'].cast(pl.Float64))
llibres = llibres.with_columns(llibres['ratings_count'].cast(pl.Int64))
llibres = llibres.with_columns(llibres['num_pages'].cast(pl.Int64))

In [ ]:
# Filtrar generos_libros para mantener solo aquellos libros que estén en llibres
generos_libros = generos_libros.filter(generos_libros['book_id'].is_in(llibres_ids))
print(generos_libros.shape,llibres.shape)

#Ajuntem els 2 dataframes
llibres = llibres.join(generos_libros, on='book_id', how='inner')

In [238]:
llibres

language_code,average_rating,similar_books,format,num_pages,isbn13,publication_year,book_id,ratings_count,title,author_id,"fantasy, paranormal",fiction,"mystery, thriller, crime",poetry,"history, historical fiction, biography",romance,non-fiction,children,young-adult,"comics, graphic"
str,f64,str,str,i64,str,str,str,i64,str,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
null,3.4,null,null,null,"""9780850308716""",null,"""287140""",15,"""Runic Astrolog…","""149918""",0,0,0,0,0,0,3,0,0,0
null,3.98,"""199623 375749""","""Hardcover""",288,"""9780375836879""","""2006""","""89377""",6949,"""Penny from Hea…","""137561""",0,425,0,0,330,0,0,190,93,0
"""eng""",4.43,"""140185 197998 …","""Hardcover""",40,"""9780590417013""","""1995""","""89378""",1331,"""Dog Heaven""","""5411""",0,13,0,0,0,0,2,109,1,0
null,4.5,null,null,null,"""9780865165144""","""2002""","""287142""",2,"""Servius' Comme…","""166520""",0,0,0,0,0,0,0,0,0,0
null,4.09,"""46609 416792""",null,null,"""9780553576344""",null,"""89379""",627,"""Hello from Hea…","""51308""",6,0,1,0,0,0,19,0,0,0
"""eng""",3.88,"""83169 107284 3…","""Paperback""",245,"""0345227018095""",null,"""828189""",64,"""Tarzan of the …","""10885""",507,825,11,0,64,45,0,31,36,0
null,3.66,null,null,93,"""9780749729127""",null,"""388676""",59,"""The Queen Eliz…","""10657""",0,2,0,0,0,0,0,8,0,0
null,3.81,"""219231 455901 …","""Paperback""",512,"""9780380821006""","""2003""","""89588""",1480,"""The Visitor""","""20560""",86,130,1,0,0,0,0,0,0,0
"""en-GB""",3.67,null,"""Paperback""",248,"""9780953626953""","""2001""","""257654""",27,"""Gillian of the…","""150591""",0,3,0,0,0,0,0,2,0,0


In [239]:
llibres.write_csv('books.csv')

In [ ]:
libros_leidos = libros_leidos.filter(libros_leidos['book_id'].is_in(llibres_ids))
libros_leidos.write_csv('users.csv')